# Load Library

In [ ]:
import torch

In [ ]:
torch.cuda.current_device()

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_MODE"] = "offline"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

import torch
import torch.nn as nn

import transformers
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from datasets import Dataset
from peft import LoraConfig, PeftConfig
import bitsandbytes as bnb
from trl import SFTTrainer

from sklearn.metrics import (accuracy_score,
                             classification_report,
                             confusion_matrix)
from sklearn.model_selection import train_test_split

In [ ]:
from transformers import logging

logging.set_verbosity_error()

In [ ]:
from datasets import load_dataset, DatasetDict

### SETUP Project Path

In [ ]:
PROJECT_PATH = '/project/lt200393-foullm/sam/2026/workshop/llm-finetune-workshop'

## Load Dataset

In [ ]:
data = load_dataset(f'{PROJECT_PATH}/dataset/raw_dataset/thai-sentiment/')

In [ ]:
data

In [ ]:
data['train'][0]

# Data PreProcess

In [ ]:
def generate_prompt(data_point):
    return f"""
            Analyze the sentiment of the tweet enclosed in square brackets,
            determine if it is positive or negative, and return the answer as
            the corresponding sentiment label "positive" or  "negative"

            [{data_point["text"]}] = {data_point["label"]}
            """.strip()

def generate_test_prompt(data_point):
    return f"""
            Analyze the sentiment of the tweet enclosed in square brackets,
            determine if it is positive or negative, and return the answer as
            the corresponding sentiment label "positive" or  "negative"

            [{data_point["text"]}] =

            """.strip()

In [ ]:
test = data['test'].to_pandas()
validation = data['validation'].to_pandas()
train = data['train'].to_pandas()

In [ ]:
def convert(x):
    if x==1:
        return 'positive'
    elif x==0:
        return 'negative'
    else:
        return 'none'

In [ ]:
train['label'] = train['label'].apply(lambda x: convert(x))
validation['label'] = validation['label'].apply(lambda x: convert(x))
test['label'] = test['label'].apply(lambda x: convert(x))

In [ ]:
train.head()

# Apply Prompt

In [ ]:
X_train = pd.DataFrame(train.apply(generate_prompt, axis=1),
                       columns=["text"])
X_eval = pd.DataFrame(validation.apply(generate_prompt, axis=1),
                      columns=["text"])

In [ ]:
y_true = test.label
X_test = pd.DataFrame(test.apply(generate_test_prompt, axis=1), columns=["text"])

In [ ]:
train_data = Dataset.from_pandas(X_train)
eval_data = Dataset.from_pandas(X_eval)

In [ ]:
print(train_data['text'][0])

In [ ]:
print(eval_data['text'][0])

In [ ]:
def evaluate(y_true, y_pred):

    labels = ['positive',  'negative']
    mapping = {'positive': 1, 'negative': 0, 'none':1,}
    def map_func(x):
        return mapping.get(x, 1)

    y_true = np.vectorize(map_func)(y_true)
    y_pred = np.vectorize(map_func)(y_pred)

    # Calculate accuracy
    accuracy = accuracy_score(y_true=y_true, y_pred=y_pred)
    print(f'Accuracy: {accuracy:.3f}')

    # Generate accuracy report
    unique_labels = set(y_true)  # Get unique labels

    for label in unique_labels:
        label_indices = [i for i in range(len(y_true))
                         if y_true[i] == label]
        label_y_true = [y_true[i] for i in label_indices]
        label_y_pred = [y_pred[i] for i in label_indices]
        accuracy = accuracy_score(label_y_true, label_y_pred)
        print(f'Accuracy for label {label}: {accuracy:.3f}')

    # Generate classification report
    class_report = classification_report(y_true=y_true, y_pred=y_pred)
    print('\nClassification Report:')
    print(class_report)

    # Generate confusion matrix
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred, labels=[0, 1])
    print('\nConfusion Matrix:')
    print(conf_matrix)

In [ ]:
model_name = f"{PROJECT_PATH}/model/qwen3-4b" #Qwen/Qwen3-1.7B #Qwen/Qwen2.5-1.5B

compute_dtype = getattr(torch, "bfloat16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
model

# Create Predict Function

In [ ]:
def inference(prompt, model=model, tokenizer=tokenizer):
    messages = [
    {"role": "user", "content": prompt+'/no_think'}
    ]  
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=100,
        temperature=0.001
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    
    return content.strip()

In [ ]:
def report(actual_y, predict_y):
    labels = ["negative", "neutral", "positive"]
    accuracy = accuracy_score(y_true=actual_y, y_pred=predict_y)
    print(f'Accuracy:\n{accuracy:.3f}')
    class_report = classification_report(y_true=actual_y, y_pred=predict_y)
    print(f'\nClassification Report:\n{class_report}')
    conf_matrix = confusion_matrix(y_true=actual_y, y_pred=predict_y, labels=labels)
    print(f'\nConfusion Matrix:\n{conf_matrix}')

In [ ]:
X_test

In [ ]:
y_true.tolist()

In [ ]:
predict_y = []
actual_y = []

limits = 10
counter = 0
for row, ans in tqdm(zip(X_test['text'].tolist(), y_true.tolist())):
    predict_y.append(inference(row, model, tokenizer))
    actual_y.append(ans)
    counter += 1
    if counter == limits:
        break

In [ ]:
predict_y[:5]

In [ ]:
y_pred = []
for row in predict_y:
    if "positive" in row:
        y_pred.append("positive")
    elif "negative" in row:
        y_pred.append("negative")
    else:
        y_pred.append("neutral")

In [ ]:
y_pred

In [ ]:
evaluate(actual_y, y_pred)

# Inference

In [ ]:
def inference_without_prompt(text, model, tokenizer):
    prompt = f"""
            Analyze the sentiment of the tweet enclosed in square brackets,
            determine if it is positive or negative, and return the answer as
            the corresponding sentiment label "positive" or  "negative"

            [{text}] =

            """.strip()
    messages = [
    {"role": "user", "content": prompt+'/no_think'}
    ] 
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=100,
        temperature=0.001
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    
    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
    answer = content.split("=")[-1].lower().strip()
    
    return answer.strip()

In [ ]:
inference_without_prompt('แฟนบอกว่าไม่เป็นไรหรอก คิดมาก', model, tokenizer)

In [ ]:
train_data

# FineTune with LoRa

In [ ]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

training_arguments = TrainingArguments(
    output_dir="logs",
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=False,
    optim="paged_adamw_8bit",
    save_steps=0,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="cosine",
    do_eval=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    peft_config=peft_config,
    args=training_arguments,
)

In [ ]:
model

In [ ]:
import os
os.environ["WANDB_MODE"] = "offline"

In [ ]:
trainer.train()

# Evaluate

In [ ]:
def predict(X_test, model, tokenizer):
    y_pred = []
    for i in tqdm(range(len(X_test))):
        prompt = X_test.iloc[i]["text"]
        input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")



        
        outputs = model.generate(**input_ids, max_new_tokens=2048, temperature=0.001,
                                 pad_token_id=tokenizer.eos_token_id, do_sample=False)
        result = tokenizer.decode(outputs[0])
        answer = result.split("=")[-1].lower()
        if "positive" in answer:
            y_pred.append("positive")
        elif "negative" in answer:
            y_pred.append("negative")
        elif "neutral" in answer:
            y_pred.append("neutral")
        else:
            y_pred.append("none")
    return y_pred

In [ ]:
y_pred = predict(X_test, model, tokenizer)
evaluate(y_true, y_pred)

In [ ]:
inference_without_prompt('แฟนบอกว่าไม่เป็นไรหรอก คิดมาก', model, tokenizer)

# Save Adapter

In [ ]:
trainer.model.save_pretrained(f"{PROJECT_PATH}/trained_model/adapter-sentiment")